In [1]:
import pandas as pd
import json
from pathlib import Path
import torch
import pickle

from medical_rag.config import MEDICAL_PATTERNS, MEDICAL_SYNONYMS
from medical_rag.chroma_indexer import PubMedChromaIndexer
from medical_rag.query_processor import MedicalQueryProcessor
from medical_rag.multipathretriever import MultiPathRetriever
from medical_rag.reranker import MedicalBGECrossEncoderReranker
from medical_rag.multiretrieval_pipeline import MedicalHybridRetrievalPipeline
from medical_rag.retriever import (
    MedicalChromaRetriever,
    metadata_matches,
    MedicalBM25Retriever
)
from medical_rag.mesh_terminology import (
    MeshLexicon,
    translate_known_medical_terms,
)

e:\anaconda3\envs\medrag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 初始化医学查询器各部分

In [2]:
#加载已知期刊列表
chunks_df = pd.read_parquet(
    "F:\RAG\data\pubmed_fulltext_chunks_2.parquet"
)

journal_series = (
    chunks_df["journal"]
    .fillna("")
    .astype(str)
    .str.strip()
)

journal_series = journal_series[
    journal_series.ne("")
]

journal_counts = (
    journal_series
    .value_counts()
)

canonical_journal_map = {}

# value_counts 已按频率从高到低排序
for journal_name in journal_counts.index:
    normalized = journal_name.casefold()

    if normalized not in canonical_journal_map:
        canonical_journal_map[
            normalized
        ] = journal_name

known_journals = list(
    canonical_journal_map.values()
)


In [3]:
#初始化查询处理器与Retriever

mesh_lexicon = MeshLexicon(
    alias_index_path=r"F:\RAG\data\MedicalTerminology\mesh\mesh_alias_index.json",
    concepts_path=r"F:\RAG\data\MedicalTerminology\mesh\mesh_concepts.jsonl",
)

query_processor = (
    MedicalQueryProcessor(
        synonyms=MEDICAL_SYNONYMS,
        patterns=MEDICAL_PATTERNS,
        corpus_language="en",
        mesh_lexicon=mesh_lexicon
    )
)

indexer = PubMedChromaIndexer(
    model_name=(
        "BAAI/bge-small-en-v1.5"
    ),
    persist_directory=(
        r"F:\RAG\vector_db\chroma_pubmed_db_rebuilt"
    ),
    collection_name=(
        "pubmed_fulltext_bge_small"
    ),
    device="cuda",
    embedding_batch_size=32,
    insert_batch_size=1000
)

indexer.collection = (
    indexer.client.get_collection(
        name=indexer.collection_name
    )
)

print(
    "Indexed vectors:",
    indexer.collection.count()
)

Loaded 247,002 MeSH aliases
Loaded 31,110 MeSH concepts
Loading embedding model: BAAI/bge-small-en-v1.5
Device: cuda
Persist directory: F:\RAG\vector_db\chroma_pubmed_db_rebuilt
Collection name: pubmed_fulltext_bge_small


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2199.38it/s]
F:\RAG\src\medical_rag\chroma_indexer.py:109: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.embedding_model.get_sentence_embedding_dimension()


Existing collections: ['pubmed_fulltext_bge_small']
Collection loaded successfully. Count: 631,000
Embedding dimension: 384
Distance metric: cosine
Indexed vectors: 631000


In [4]:
retriever = MedicalChromaRetriever(
    query_processor=query_processor,
    indexer=indexer,
    known_journals=known_journals,
    query_translator=translate_known_medical_terms
)

# 多路检索

## 构建BM25索引

In [5]:
bm25_retriever = MedicalBM25Retriever(
    index_directory=(
        r"F:\RAG\vector_db\pubmed_bm25_index"
    )
)

bm25_stats = bm25_retriever.build(
    chunks_df=chunks_df,
    chroma_collection=indexer.collection,
    align_with_chroma=True,
)

print(json.dumps(
    bm25_stats,
    ensure_ascii=False,
    indent=4,
))

Reading Chroma IDs: 100%|██████████| 13/13 [00:19<00:00,  1.49s/it]


Original chunks: 668,461
BM25 indexed chunks: 631,000


{
    "index_type": "BM25S",
    "original_chunks": 668461,
    "indexed_chunks": 631000,
    "aligned_with_chroma": true,
    "built_at": "2026-08-19T20:40:55.747895",
    "index_directory": "F:\\RAG\\vector_db\\pubmed_bm25_index"
}


In [6]:
#加载生成的BM25索引

bm25_retriever = MedicalBM25Retriever(
    index_directory=(
        r"F:\RAG\vector_db\pubmed_bm25_index"
    )
).load(mmap=True)

BM25 loaded: 631,000 chunks


## MultiPathRetriever

In [7]:
#检索初始化

multi_path_retriever = MultiPathRetriever(
    query_processor=query_processor,
    vector_indexer=indexer,
    medical_retriever=retriever,
    bm25_retriever=bm25_retriever,
    vector_weight=0.65,
    bm25_weight=0.35,
    rrf_k=60,
)

print("MultiPathRetriever initialized")
print(
    "Chroma vectors:",
    indexer.collection.count()
)
print(
    "BM25 documents:",
    len(bm25_retriever.mapping_df)
)

MultiPathRetriever initialized
Chroma vectors: 631000
BM25 documents: 631000


In [8]:
#验证索引数量是否一致

assert (
    indexer.collection.count()
    == len(bm25_retriever.mapping_df)
), (
    "Chroma和BM25索引数量不一致，"
    "建议重新对齐BM25文档集合。"
)

## 测试多路检查

In [9]:
search_output = (
    multi_path_retriever.retrieve(
        query=(
            "检索2021年以来"
            "Nature Communications中"
            "关于metformin的研究"
        ),
        top_k_vector=50,
        top_k_keyword=50,
        top_k_fused=20,
        fusion_strategy="rrf",
        max_chunks_per_doc=2,
    )
)

print(
    json.dumps(
        search_output["statistics"],
        ensure_ascii=False,
        indent=4,
    )
)

{
    "vector_count": 50,
    "bm25_count": 3,
    "merged_candidate_count": 53,
    "fused_count": 20,
    "fusion_strategy": "rrf"
}


In [10]:
#查看查询增强结果

query_info = search_output["query_info"]

print("Vector query:")
print(query_info["vector_query"])

print("\nKeyword terms:")
print(query_info["keyword_terms"])

print("\nMetadata filter:")
print(query_info["where_filter"])

Vector query:
检索2021年以来Nature Communications中关于metformin的研究 Related medical concepts: dimethylbiguanide; Glucophage; D008687; D019368.

Keyword terms:
['metformin', 'dimethylbiguanide', 'Glucophage', 'D008687', 'nature', 'D019368']

Metadata filter:
{'$and': [{'journal': 'Nature Communications'}, {'publication_year': {'$gte': 2021}}, {'publication_year': {'$lte': 2026}}]}


In [11]:
#查看向量查询结果
display(
    search_output["vector_results"][
        [
            "vector_rank",
            "similarity",
            "source_title",
            "journal",
            "publication_year",
            "pmid",
        ]
    ].head(10)
)

,vector_rank,similarity,source_title,journal,publication_year,pmid
0,1,0.676239,A Pilot randomized trial to examine effects of...,Nature Communications,2022,
1,2,0.674050,Bariatric surgery induces a new gastric mucosa...,Nature Communications,2021,33397977
2,3,0.673936,Bariatric surgery induces a new gastric mucosa...,Nature Communications,2021,33397977
3,4,0.673138,Bariatric surgery induces a new gastric mucosa...,Nature Communications,2021,33397977
4,5,0.672013,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,
5,6,0.670476,Bariatric surgery induces a new gastric mucosa...,Nature Communications,2021,33397977
6,7,0.669880,Lure-and-kill macrophage nanoparticles allevia...,Nature Communications,2021,34230486
7,8,0.667153,Reversal of the renal hyperglycemic memory in ...,Nature Communications,2022,36030260
8,9,0.665311,A Pilot randomized trial to examine effects of...,Nature Communications,2022,
9,10,0.664859,Lure-and-kill macrophage nanoparticles allevia...,Nature Communications,2021,34230486


In [12]:
#查看BM25查询结果
display(
    search_output["bm25_results"][
        [
            "bm25_rank",
            "bm25_score",
            "source_title",
            "journal",
            "publication_year",
            "pmid",
        ]
    ].head(10)
)

,bm25_rank,bm25_score,source_title,journal,publication_year,pmid
0,1,2.398165,Evidence of anthropogenic impacts on global dr...,Nature Communications,2021,33980822
1,2,2.368324,Evidence of anthropogenic impacts on global dr...,Nature Communications,2021,33980822
2,3,2.363294,Germline mutations in mitochondrial complex I ...,Nature Communications,2022,35551192


In [13]:
#查看融合后的最终结果

result_columns = [
    "fusion_rank",
    "fusion_score",
    "vector_rank",
    "similarity",
    "bm25_rank",
    "bm25_score",
    "source_title",
    "journal",
    "publication_year",
    "pmid",
    "doc_id",
    "chunk_index",
    "text",
]

available_columns = [
    column
    for column in result_columns
    if column
    in search_output[
        "fused_results"
    ].columns
]

display(
    search_output[
        "fused_results"
    ][available_columns]
)

,fusion_rank,fusion_score,vector_rank,similarity,bm25_rank,bm25_score,source_title,journal,publication_year,pmid,doc_id,chunk_index,text
0,1,0.016393,1.0,0.676239,NaN,NaN,A Pilot randomized trial to examine effects of...,Nature Communications,2022,,DOC_327ad2b049a6,2,Title: A Pilot randomized trial to examine eff...
1,2,0.016393,NaN,NaN,1.0,2.398165,Evidence of anthropogenic impacts on global dr...,Nature Communications,2021,33980822,PMID_33980822,5,Title: Evidence of anthropogenic impacts on gl...
2,3,0.016129,2.0,0.674050,NaN,NaN,Bariatric surgery induces a new gastric mucosa...,Nature Communications,2021,33397977,PMID_33397977,34,Title: Bariatric surgery induces a new gastric...
3,4,0.016129,NaN,NaN,2.0,2.368324,Evidence of anthropogenic impacts on global dr...,Nature Communications,2021,33980822,PMID_33980822,11,Title: Evidence of anthropogenic impacts on gl...
4,5,0.015873,3.0,0.673936,NaN,NaN,Bariatric surgery induces a new gastric mucosa...,Nature Communications,2021,33397977,PMID_33397977,36,"GLP-1 (1–36) amide, and GLP-1 (1–37). The anti..."
5,6,0.015873,NaN,NaN,3.0,2.363294,Germline mutations in mitochondrial complex I ...,Nature Communications,2022,35551192,PMID_35551192,33,Title: Germline mutations in mitochondrial com...
6,7,0.015385,5.0,0.672013,NaN,NaN,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,,DOC_979625960f1d,20,Title: Cutaneous and acral melanoma cross-OMIC...
7,8,0.014925,7.0,0.669880,NaN,NaN,Lure-and-kill macrophage nanoparticles allevia...,Nature Communications,2021,34230486,PMID_34230486,27,Title: Lure-and-kill macrophage nanoparticles ...
8,9,0.014706,8.0,0.667153,NaN,NaN,Reversal of the renal hyperglycemic memory in ...,Nature Communications,2022,36030260,PMID_36030260,34,Title: Reversal of the renal hyperglycemic mem...
9,10,0.014493,9.0,0.665311,NaN,NaN,A Pilot randomized trial to examine effects of...,Nature Communications,2022,,DOC_327ad2b049a6,30,Title: A Pilot randomized trial to examine eff...


## 基础验证

In [14]:
vector_df = search_output["vector_results"]
bm25_df = search_output["bm25_results"]
fused_df = search_output["fused_results"]

assert not vector_df.empty, (
    "向量检索未返回结果"
)

assert not bm25_df.empty, (
    "BM25检索未返回结果"
)

assert fused_df["vector_id"].is_unique, (
    "融合结果存在重复vector_id"
)

assert (
    fused_df["fusion_score"]
    .is_monotonic_decreasing
), "融合结果未按分数降序排列"

assert (
    fused_df.groupby("doc_id")
    .size()
    .max()
    <= 2
), "单篇文献返回了过多Chunk"

print("PASS: 向量检索正常")
print("PASS: BM25检索正常")
print("PASS: 融合去重正常")
print("PASS: 融合排序正常")
print("PASS: 文档多样化正常")

PASS: 向量检索正常
PASS: BM25检索正常
PASS: 融合去重正常
PASS: 融合排序正常
PASS: 文档多样化正常


## 初始化BGE Cross-Encoder重排序器

In [15]:
reranker = (
    MedicalBGECrossEncoderReranker(
        model_name=(
            r"F:\RAG\models"
            r"\bge-reranker-base"
        ),
        device=(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        ),
        batch_size=2,
        max_length=384,
        criteria_weights={
            "relevance": 0.60,
            "recency": 0.25,
            "authority": 0.15,
        },
        recency_window=10,
    )
)

Loading reranker: F:\RAG\models\bge-reranker-base
Reranker device: cuda


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:01<00:00, 125.30it/s]


## 完整检索Pipeline

In [16]:
#初始化
retrieval_pipeline = (
    MedicalHybridRetrievalPipeline(
        multi_path_retriever=(
            multi_path_retriever
        ),
        reranker=reranker,
    )
)

print(
    "Medical retrieval pipeline initialized"
)

Medical retrieval pipeline initialized


In [17]:
#运行完整检索
pipeline_output = (
    retrieval_pipeline.search(
        query=(
            "检索2021年以来"
            "Nature Communications中"
            "关于metformin的研究"
        ),
        top_k_vector=50,
        top_k_keyword=50,
        top_k_fused=30,
        final_top_k=10,
        fusion_strategy="rrf",
        max_chunks_per_doc=2,
        apply_reranking=True,
    )
)

print(
    json.dumps(
        pipeline_output["statistics"],
        ensure_ascii=False,
        indent=4,
        default=str,
    )
)

print(
    "Rerank query:"
)

print(
    pipeline_output["rerank_query"]
)

BGE reranking: 100%|██████████| 10/10 [00:00<00:00, 10.42it/s]

{
    "vector_count": 50,
    "bm25_count": 3,
    "merged_candidate_count": 53,
    "fused_count": 20,
    "fusion_strategy": "rrf",
    "reranking_applied": true,
    "reranker_model": "F:\\RAG\\models\\bge-reranker-base",
    "rerank_candidate_count": 20,
    "final_result_count": 10,
    "reference_year": 2026
}
Rerank query:
检索2021年以来Nature Communications中关于metformin的研究 Related medical concepts: dimethylbiguanide; Glucophage; D008687; D019368.


## 展示最终结果

In [18]:
final_results = (
    pipeline_output["final_results"]
)

display_columns = [
    "final_rank",
    "final_score",
    "relevance_score",
    "recency_score",
    "authority_score",
    "fusion_rank",
    "fusion_score",
    "vector_rank",
    "similarity",
    "bm25_rank",
    "bm25_score",
    "source_title",
    "journal",
    "publication_year",
    "pmid",
    "doc_id",
    "chunk_index",
    "text",
]

available_columns = [
    column
    for column in display_columns
    if column in final_results.columns
]

display(
    final_results[available_columns]
)

,final_rank,final_score,relevance_score,recency_score,authority_score,fusion_rank,fusion_score,vector_rank,similarity,bm25_rank,bm25_score,source_title,journal,publication_year,pmid,doc_id,chunk_index,text
0,1,0.283625,0.010209,0.6,0.85,10,0.014493,9.0,0.665311,NaN,NaN,A Pilot randomized trial to examine effects of...,Nature Communications,2022,,DOC_327ad2b049a6,30,Title: A Pilot randomized trial to examine eff...
1,2,0.282666,0.008611,0.6,0.85,19,0.009434,46.0,0.653764,NaN,NaN,The gut hormone Allatostatin C/Somatostatin re...,Nature Communications,2022,35121731,PMID_35121731,76,Title: The gut hormone Allatostatin C/Somatost...
2,3,0.281840,0.007233,0.6,0.85,15,0.013158,16.0,0.662351,NaN,NaN,Reversal of the renal hyperglycemic memory in ...,Nature Communications,2022,36030260,PMID_36030260,41,Title: Reversal of the renal hyperglycemic mem...
3,4,0.280644,0.005240,0.6,0.85,17,0.010526,35.0,0.657789,NaN,NaN,The gut hormone Allatostatin C/Somatostatin re...,Nature Communications,2022,35121731,PMID_35121731,82,Title: The gut hormone Allatostatin C/Somatost...
4,5,0.280501,0.005002,0.6,0.85,9,0.014706,8.0,0.667153,NaN,NaN,Reversal of the renal hyperglycemic memory in ...,Nature Communications,2022,36030260,PMID_36030260,34,Title: Reversal of the renal hyperglycemic mem...
5,6,0.279331,0.003052,0.6,0.85,1,0.016393,1.0,0.676239,NaN,NaN,A Pilot randomized trial to examine effects of...,Nature Communications,2022,,DOC_327ad2b049a6,2,Title: A Pilot randomized trial to examine eff...
6,7,0.278938,0.002397,0.6,0.85,6,0.015873,NaN,NaN,3.0,2.363294,Germline mutations in mitochondrial complex I ...,Nature Communications,2022,35551192,PMID_35551192,33,Title: Germline mutations in mitochondrial com...
7,8,0.277837,0.000561,0.6,0.85,20,0.009346,47.0,0.653515,NaN,NaN,Sympathetic axonal sprouting induces changes i...,Nature Communications,2022,35418199,PMID_35418199,78,Title: Sympathetic axonal sprouting induces ch...
8,9,0.277819,0.000532,0.6,0.85,16,0.011628,26.0,0.659992,NaN,NaN,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,,DOC_979625960f1d,17,"vitiligo (BCH and TCGA), melanoma (BCH), and s..."
9,10,0.277614,0.000190,0.6,0.85,7,0.015385,5.0,0.672013,NaN,NaN,Cutaneous and acral melanoma cross-OMICs revea...,Nature Communications,2022,,DOC_979625960f1d,20,Title: Cutaneous and acral melanoma cross-OMIC...


## Pipeline质量验证

In [19]:
#基础结果验证

vector_df = pipeline_output[
    "vector_results"
]

bm25_df = pipeline_output[
    "bm25_results"
]

fused_df = pipeline_output[
    "fused_results"
]

final_df = pipeline_output[
    "final_results"
]

assert not vector_df.empty, (
    "向量召回结果为空"
)

assert not bm25_df.empty, (
    "BM25召回结果为空"
)

assert not fused_df.empty, (
    "融合结果为空"
)

assert not final_df.empty, (
    "重排结果为空"
)

print("PASS: 所有检索阶段均返回结果")

PASS: 所有检索阶段均返回结果


In [20]:
#去重和排序验证

assert fused_df["vector_id"].is_unique, (
    "融合候选存在重复vector_id"
)

assert final_df["vector_id"].is_unique, (
    "最终结果存在重复vector_id"
)

assert (
    final_df["final_score"]
    .is_monotonic_decreasing
), "最终结果未按final_score降序排列"

print("PASS: 去重正常")
print("PASS: 最终排序正常")

PASS: 去重正常
PASS: 最终排序正常


In [21]:
#分数范围验证

score_columns = [
    "relevance_score",
    "recency_score",
    "authority_score",
    "final_score",
]

for column in score_columns:
    assert (
        final_df[column]
        .between(0.0, 1.0)
        .all()
    ), f"{column}存在超出[0,1]的值"

print("PASS: 多准则分数范围正常")

PASS: 多准则分数范围正常


In [22]:
#元数据过滤验证
where_filter = (
    pipeline_output[
        "query_info"
    ].get("where_filter")
)

if where_filter:
    invalid_rows = final_df[
        ~final_df.apply(
            lambda row: metadata_matches(
                row,
                where_filter,
            ),
            axis=1,
        )
    ]

    assert invalid_rows.empty, (
        "最终结果中存在不符合"
        "Metadata Filter的记录"
    )

    print(
        "PASS: 最终结果全部满足"
        "Metadata Filter"
    )
else:
    print(
        "INFO: 当前查询没有元数据过滤条件"
    )

PASS: 最终结果全部满足Metadata Filter


In [23]:
#文档级多样性验证

if "doc_id" in final_df.columns:
    maximum_chunks = (
        final_df.groupby("doc_id")
        .size()
        .max()
    )

    assert maximum_chunks <= 2, (
        "单篇文献返回了超过2个Chunk"
    )

    print(
        "PASS: 文档级多样性正常"
    )

PASS: 文档级多样性正常


## 保存Pipeline_output

In [24]:
output_directory = Path(
    r"F:\RAG\data\pipeline_outputs"
)

output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

output_path = (
    output_directory
    / "task6_pipeline_output.pkl"
)

with output_path.open("wb") as file:
    pickle.dump(
        pipeline_output,
        file,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print(
    "pipeline_output保存完成：",
    output_path,
)

print(
    "文件大小：",
    round(
        output_path.stat().st_size
        / 1024
        / 1024,
        2,
    ),
    "MB",
)

pipeline_output保存完成： F:\RAG\data\pipeline_outputs\task6_pipeline_output.pkl
文件大小： 0.12 MB
